# Stuff Document Chain

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

![](https://publish-01.obsidian.md/access/84bff78007c07f71220692232b80322e/Study/LangChain/Reference/RAG%20Reference/Stufff%20Chain%20Process.jpg)

```
[문서1] + [문서2] + [문서3] + [질문] → LLM → 답변
```

검색된 모든 문서를 하나의 프롬프트에 그대로 집어넣어 한 번의 LLM 호출로 답변을 생성합니다.

| 장점 | 단점 |
|------|------|
| 구현이 단순하고 빠름 | 문서가 많으면 컨텍스트 길이 초과 |
| LLM이 전체 맥락을 한번에 파악 | 토큰 비용 증가 |
| 가장 낮은 레이턴시 | 긴 문서에 부적합 |

> 소량의 짧은 문서에 최적. **가장 기본적인 방식**
> 

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model='gpt-4o')

prompt = ChatPromptTemplate.from_template("""
다음 제공된 문맥(context)만을 사용하여 질문에 답하세요:
{context}

질문: {question}

""")

# Retrieve 된 Document 데이터 준비
docs = [
    Document(page_content="사과는 보라색 과일입니다"),
    Document(page_content="바나나는 노란색 과일입니다"),
]


# Retrieve 된 Document 들을 하나로 합치는 함수 정의 (stuff 로직)
def format_docs(docs):
    # Document 들의 page_content 를 추출하여 "\n\n" 로 연결
    return "\n\n".join(doc.page_content for doc in docs)


chain = (
    {
        "context": lambda x: format_docs(x['documents']), 
        "question": lambda x: x['question'],
    }   
    | prompt 
    | llm
)

response = chain.invoke({
    "documents": docs,
    "question": "사과와 바나나는 각각 무슨 색인가요?",
})

print(response.content)






사과는 보라색이고, 바나나는 노란색입니다.


# Map-Reduce chain
**"각 문서를 개별 요약 후 최종 합산"**

![](https://publish-01.obsidian.md/access/84bff78007c07f71220692232b80322e/Study/LangChain/Reference/RAG%20Reference/Map%20Reduce%20Chain%20Process.jpg)

```
[문서1] → LLM → 중간답변1 ┐
[문서2] → LLM → 중간답변2 ┼→ LLM → 최종 답변
[문서3] → LLM → 중간답변3 ┘
```

**Map 단계**: 각 문서에 대해 독립적으로 LLM을 호출해 중간 답변 생성  
**Reduce 단계**: 모든 중간 답변을 합쳐 최종 답변 생성

| 장점 | 단점 |
|------|------|
| 문서 수에 제한 없음 | LLM 호출 횟수 많음 (비용↑) |
| 병렬 처리 가능 (Map 단계) | 레이턴시 높음 |
| 대용량 문서에 적합 | 문서 간 연관성 파악 어려움 |

> 문서가 많고 병렬 처리가 가능할 때 사용

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap, RunnableLambda
from langchain_openai import ChatOpenAI
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

In [4]:
# LLM 정의
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# --------------------
# 1. MAP 단계 Prompt
# --------------------

map_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 문서를 요약하는 AI입니다."),
    ("human", "다음 문서를 핵심만 요약하세요:\n\n{document}"),
])

map_chain = map_prompt | llm | StrOutputParser()

# --------------------
# 2. REDUCE 단계 Prompt
# --------------------

reduce_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 여러 요약을 하나의 요약으로 통합하는 AI입니다."),
    ("human", "다음 요약들을 하나의 간결한 요약으로 통합하세요:\n\n{summaries}")    
])

reduce_chain = reduce_prompt | llm | StrOutputParser()

# --------------------
# 3. Map-Reduce LCEL 체인
# --------------------
map_reduce_chain = (
    #  Map 단계
    RunnableLambda(
        lambda docs: [map_chain.invoke({"document": doc.page_content}) for doc in docs]
    )
    # Reduce 단계
    | RunnableLambda(
        lambda summaries: reduce_chain.invoke({'summaries': '\n\n'.join(summaries)})
    )
)

# --------------------
# 4. 테스트용 문서 준비
# --------------------
documents = [
    Document(page_content="LangChain은 LLM 애플리케이션을 구축하기 위한 프레임워크입니다."),
    Document(page_content="LCEL은 LangChain에서 체인을 선언적으로 구성할 수 있도록 도와줍니다."),
    Document(page_content="Map-Reduce 체인은 대규모 문서를 효율적으로 요약하는 데 유용합니다.")
]

# --------------------
# 5. 실행
# ----------------------
result = map_reduce_chain.invoke(documents)
print(result)




LangChain은 대형 언어 모델(LLM) 애플리케이션 개발을 위한 프레임워크로, LCEL을 통해 체인을 선언적으로 구성할 수 있으며, Map-Reduce 체인은 대규모 문서의 효율적인 요약에 유용한 방법입니다.


# Map-Rerank Chain
**"각 문서로 답변 생성 후 가장 확신 높은(Score 가 높은) 답변 선택"**

![](https://publish-01.obsidian.md/access/84bff78007c07f71220692232b80322e/Study/LangChain/Reference/RAG%20Reference/Map%20Re-rank%20Process.png)

```
[문서1] → LLM → (답변1, 확신도: 90%) ┐
[문서2] → LLM → (답변2, 확신도: 60%) ┼→ 최고 점수 선택 → 최종 답변
[문서3] → LLM → (답변3, 확신도: 75%) ┘
```

각 문서마다 LLM이 답변과 함께 **확신도 점수**를 반환하고, 가장 높은 점수의 답변을 최종 선택합니다.

| 장점 | 단점 |
|------|------|
| 가장 관련성 높은 답변 선택 | LLM 호출 횟수 많음 |
| 노이즈 문서 영향 최소화 | 확신도 점수가 항상 정확하지 않음 |
| 단일 정답이 명확할 때 유리 | 문서 간 종합이 필요한 경우 부적합 |

> 정답이 특정 문서 하나에 집중되어 있을 때 유리


In [10]:
# Map 단계

map_prompt = ChatPromptTemplate.from_template(
    """
    너는 질문-응답 평가 모델이다.

    질문:
    {question}

    문서:
    {context}

    위 문서를 기반으로 질문에 답하라.
    그리고 답변의 질문 관련성을 0~100 사이의 점수로 평가하라.

    반드시 아래 형식으로만 출력하라:
    score: <점수>
    answer: <답변>
    """
)

map_chain = map_prompt | llm | StrOutputParser()

documents = [
    Document(page_content="LangChain은 LLM 애플리케이션 프레임워크이다."),
    Document(page_content="LCEL은 LangChain의 선언형 체인 문법이다."),
    Document(page_content="Map-Rerank는 문서별 점수를 비교하는 체인이다.")
]

question = "LangChain의 LCEL은 무엇인가?"

map_results = [
    map_chain.invoke({
        "question": question,
        "context": doc.page_content,
    })
    for doc in documents
]

print(map_results) # 확인

def parse_result(text):
    import re
    score_match = re.search(r"score:\s*(\d+)", text)
    answer_match = re.search(r"answer:\s*(.*)", text, re.DOTALL)  #  re.DOTALL 는 "." 이 줄바꿈 문자(\n)도 포함해서 매칭

    return {
        "score": int(score_match.group(1)),
        "answer": answer_match.group(1).strip(),
    }

# Re-rank 단계
parsed_results = [
    # {
    #     "score": ...
    #     "answer": ...
    # }
    parse_result(r)
    for r in map_results
]

# print(parsed_results) # 확인결과

best_result = max(parsed_results, key=lambda x: x['score'])

print("✅ 최종 선택된 답변")
print("Score:", best_result["score"])
print("Answer:", best_result["answer"])


['score: 20  \nanswer: LangChain은 LLM 애플리케이션 프레임워크이며, LCEL에 대한 정보는 제공되지 않았습니다.', 'score: 100  \nanswer: LCEL은 LangChain의 선언형 체인 문법입니다.', 'score: 0  \nanswer: 문서에서 LangChain의 LCEL에 대한 정보는 제공되지 않았습니다. 따라서 질문에 대한 답변을 할 수 없습니다.']
✅ 최종 선택된 답변
Score: 100
Answer: LCEL은 LangChain의 선언형 체인 문법입니다.


# Refine Document Chain
**"문서를 순차적으로 읽으며 답변을 점진적으로 개선"**

![](https://publish-01.obsidian.md/access/84bff78007c07f71220692232b80322e/Study/LangChain/Reference/RAG%20Reference/Refine%20Chain%20Process.jpg)

```
[문서1] → LLM → 초안 답변
         ↓
[문서2] + 초안 → LLM → 개선된 답변
                  ↓
         [문서3] + 개선된 답변 → LLM → 최종 답변
```

첫 번째 문서로 초안을 만들고, 이후 문서를 하나씩 읽으면서 답변을 지속적으로 **정제(Refine)**합니다.

| 장점 | 단점 |
|------|------|
| 문서 간 정보를 유기적으로 통합 | 순차 처리라 병렬화 불가 |
| 복잡한 질문에 높은 품질 | 레이턴시가 가장 높음 |
| 컨텍스트를 점진적으로 축적 | 초반 문서가 결과에 더 큰 영향 |

> 여러 문서의 정보를 종합해야 하는 복잡한 질문에 적합


In [13]:
from langchain_core.prompts import PromptTemplate

# 초기 답변용 프롬프트
initial_prompt = PromptTemplate.from_template(
    """
    다음 문서를 기반으로 질문에 대한 답변을 작성하세요.

    문서:
    {context}

    질문:
    {question}

    답변:
    """
)

# Refine 프롬프트
refine_prompt = PromptTemplate.from_template(
    """
    기존 답변:
    {existing_answer}

    아래 문서를 참고하여 기존 답변을 개선하세요.
    만약 도움이 되지 않으면 기존 답변을 그대로 유지하세요.

    문서:
    {context}

    질문:
    {question}

    개선된 답변:
    """
)

# 초기 체인 (첫번째 문서용)
initial_chain = initial_prompt | llm | StrOutputParser()

# Refine 체인 (두번째 문서부터)
refine_chain = refine_prompt | llm | StrOutputParser()


# Refine 로직
def refine_documents(inputs):
    docs = inputs['documents']
    question = inputs['question']

    # 첫 문서 -> 초기답변
    answer = initial_chain.invoke({
        "context": docs[0].page_content,
        "question": question,
    })

    # 나머지 문서들 refine 
    for doc in docs[1:]:  # 두번째 문서부터!
        answer = refine_chain.invoke({  # 기존의 답변 업데이트
            "existing_answer": answer,   # 기존의 답변 전달
            "context": doc.page_content,
            "question": question,
        })

    return answer # 최종 답변 리턴
    

refine_documents_chain = RunnableLambda(refine_documents)


# 입력 Document 들
documents = [
    Document(page_content="LangChain은 LLM 애플리케이션을 구축하기 위한 프레임워크이다."),
    Document(page_content="LCEL은 체인을 선언적으로 구성할 수 있게 해준다."),
    Document(page_content="Refine 체인은 문서를 순차적으로 처리하며 답변을 개선한다.")
]

# refine-document 체인 호출
result = refine_documents_chain.invoke({
    "documents": documents,
    "question": "LangChain에서 Refine 체인이 무엇인지 설명해줘"
})

print('✅', result)


✅ LangChain에서 Refine 체인은 주어진 입력 데이터를 순차적으로 처리하여 답변을 개선하는 과정을 포함하는 체인입니다. 이 체인은 원래의 데이터를 기반으로 추가적인 정보를 생성하거나, 데이터를 정제하여 더 나은 결과를 도출하는 데 사용됩니다. Refine 체인은 데이터의 품질을 높이고, 최종 결과물이 더 유용하고 정확하도록 돕는 역할을 합니다. 이를 통해 LLM 애플리케이션의 성능을 향상시킬 수 있습니다. 또한, LangChain의 LCEL(Chain Expression Language)을 활용하면 이러한 Refine 체인을 선언적으로 구성할 수 있어, 개발자가 체인을 보다 직관적으로 설계하고 관리할 수 있습니다.


# 정리

| | Stuff | Map-Reduce | Map-Rerank | Refine |
|---|---|---|---|---|
| **LLM 호출 수** | 1회 | N+1회 | N회 | N회 |
| **병렬 처리** | - | ✅ | ✅ | ❌ |
| **대용량 문서** | ❌ | ✅ | ✅ | ✅ |
| **문서 간 통합** | ✅ | 부분적 | ❌ | ✅ |
| **속도** | 가장 빠름 | 중간 | 중간 | 가장 느림 |
| **추천 상황** | 소량·단순 QA | 대량 문서 요약 | 단일 정답 검색 | 복잡한 종합 질문 |

**실무에서는** 대부분 **Stuff** → 문서가 많으면 **Map-Reduce** 또는 **Refine** 순으로 고려하는 것이 일반적입니다.